In [1]:
# Load only from python 3.12
import sys

sys.path = [p for p in sys.path if "python3.8" not in p]

In [16]:
!{sys.executable} -m pip install pyarrow==23.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 45.4 MB/s  0:00:01m0:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  You can safely remove it manually.


In [2]:
import geopandas as gpd
import dask_geopandas as dgpd
import dask.dataframe as dd
from dask.distributed import Client
from shapely.geometry import box
import numpy as np
import pandas as pd

In [ ]:
def main():
    # 1. Start the Dask Cluster
    # Calling Client() automatically detects all 72 cores and sets them up as workers.
    client = Client()
    print(f"Cluster active. Dashboard link: {client.dashboard_link}")
    print(f"Using {sum(client.nthreads().values())} CPU cores.")

    # 2. Load and prep the boundary (happens quickly on the main thread)
    print("Loading Europe boundary...")
    boundary = gpd.read_file("data/europe.geojson")
    
    if boundary.crs != "EPSG:3035":
        boundary = boundary.to_crs("EPSG:3035")
        
    minx, miny, maxx, maxy = boundary.total_bounds
    grid_size = 500

    # 3. Generate the bottom-left X/Y coordinates for every possible square
    print("Calculating grid coordinate pairs...")
    x_coords = np.arange(minx, maxx, grid_size)
    y_coords = np.arange(miny, maxy, grid_size)
    xx, yy = np.meshgrid(x_coords, y_coords)
    
    # Store them in a standard Pandas dataframe
    df = pd.DataFrame({'x': xx.flatten(), 'y': yy.flatten()})

    # 4. THE MAGIC: Convert to a Dask DataFrame and split it into chunks
    # A good rule of thumb is 2-3 partitions per CPU core. 72 * 2 = 144 chunks.
    print("Partitioning data for parallel processing...")
    ddf = dd.from_pandas(df, npartitions=144)

    # 5. Define the function each CPU will run on its assigned chunk
    def make_polygons(df_chunk):
        # Create the 500x500 box for this specific chunk of coordinates
        polys = [box(row.x, row.y, row.x + grid_size, row.y + grid_size) for _, row in df_chunk.iterrows()]
        return gpd.GeoDataFrame({'geometry': polys}, crs="EPSG:3035")

    # 6. Map the function to the cores. 
    # (meta tells Dask what the output format will look like before it runs)
    meta_gdf = gpd.GeoDataFrame({'geometry': []}, crs="EPSG:3035")
    dgdf = dgpd.from_dask_dataframe(
        ddf.map_partitions(make_polygons, meta=meta_gdf)
    )

    # 7. Set up the parallel spatial join (filtering out the ocean)
    print("Distributing the spatial join across all CPUs...")
    # This matches the Dask grid against the standard boundary
    filtered_dgdf = dgpd.sjoin(dgdf, boundary, how='inner', predicate='intersects')

    # 8. EXECUTE: Up until now, Dask was just planning. .compute() fires the starting gun.
    print("Executing computation... This is where the 72 CPUs go to work.")
    final_gdf = filtered_dgdf.compute()

    # Clean up extra columns from the join
    final_gdf = final_gdf[['geometry']]

    # 9. Save the output
    print(f"Saving {len(final_gdf)} grid cells to GeoPackage...")
    final_gdf.to_file("data/europe_500m_grid.gpkg", driver="GPKG")
    print("Done!")

if __name__ == '__main__':
    # This block is required when using Dask multiprocessing
    main()

Cluster active. Dashboard link: http://127.0.0.1:8787/status
Using 72 CPU cores.
Loading Europe boundary...
Calculating grid coordinate pairs...
Partitioning data for parallel processing...
Distributing the spatial join across all CPUs...
Executing computation... This is where the 72 CPUs go to work.


/home/jovyan/miniconda3/envs/parking/lib/python3.12/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 5.53 GiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [6]:
boundary = gpd.read_file("data/europe.geojson")

# Get the absolute limits (bounding box) of Europe
minx, miny, maxx, maxy = boundary.total_bounds

grid_size = 500

In [7]:
# Generate the grid coordinates
x_coords = np.arange(minx, maxx, grid_size)
y_coords = np.arange(miny, maxy, grid_size)

In [ ]:
# Create the polygon geometries
print("Building grid cells... (This may take a minute for all of Europe)")
polygons = [box(x, y, x + grid_size, y + grid_size) 
            for x in x_coords 
            for y in y_coords]

In [ ]:
# Convert to gdf
grid = gpd.GeoDataFrame({'geometry': polygons}, crs="EPSG:3035")

In [ ]:
# Filter out empty space with spatial join
final_grid = gpd.sjoin(grid, boundary, how='inner', predicate='intersects')

# Clean up the extra columns added by the join
final_grid = final_grid[['geometry']] 

In [ ]:
# Save
final_grid.to_file("data/europe_500m_grid.gpkg", driver="GPKG")